# CabbageGuard → TFLite (GreenBidder)

Converts the pre-trained `final.keras` into a mobile-ready `cabbageguard.tflite` plus the exact contract files the app reads at runtime.

**Upload these files first (File → Upload files):**
1. `final.keras` — the CabbageGuard model
2. `cabbageguard_to_tflite.py` — the pipeline script from `docs/cabbage-feature/colab/`
3. *optional:* `class_names.json` — JSON list of the 8 class keys in index order (if you have it; otherwise fill `CLASSES` below)
4. *optional:* `test_images/` — 5–10 sample cabbage images (one per class if possible) for the Keras/TFLite agreement check

**Run the cells in order.** The script refuses to ship a model with an unverified class order or preprocessing — if it stops, read the STOP message and fix the input it asks for.

## 1. Environment


In [ ]:
# Pin TF to a version known to convert EfficientNetV2 cleanly; 2.15/2.17 also work.
!pip install -q "tensorflow==2.16.1" pillow numpy

import sys, tensorflow as tf
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Upload `final.keras`

In [ ]:
from google.colab import files
import os

uploaded = files.upload()  # click, choose final.keras (and class_names.json if you have it)
print("Uploaded:", list(uploaded.keys()))

MODEL_PATH = None
for name in uploaded:
    if name.endswith(".keras"):
        MODEL_PATH = name
if MODEL_PATH is None:
    raise SystemExit("No .keras file uploaded.")
print(f"Using model: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1e6:.2f} MB)")

## 3. Configure the run

- **CLASSES** — set only if you do *not* upload `class_names.json`. The 8 class keys, in index order, comma-separated. They must match the order the model was trained with (training notebook / dataset folder order / model card). **If you are unsure, stop and check — do not guess.**
- **PREPROCESSING** — leave `None` and let the script decide empirically (it inspects the model's preprocessing layers). Only set it to `"raw_0_255"`, `"divide_255"`, or `"imagenet"` if the script tells you the decision is inconclusive.

In [ ]:
CLASSES = None
# e.g. CLASSES = "alternaria_leaf_spot,bacterial_leaf_spot,black_rot,clubroot,downy_mildew,grey_mould,healthy,ringspot"

PREPROCESSING = None  # None | "raw_0_255" | "divide_255" | "imagenet"

assert os.path.exists("cabbageguard_to_tflite.py"), "Upload cabbageguard_to_tflite.py first (docs/cabbage-feature/colab/)."
print("Config ready.")

## 4. Optional: test images for validation

Upload 5–10 cabbage images (one per class is ideal). They land in `test_images/`. The script compares Keras vs TFLite top-1 on each image and **fails the run on any mismatch**.

In [ ]:
os.makedirs("test_images", exist_ok=True)
test_files = files.upload()  # optional — you can skip this cell and re-run later
for name, content in test_files.items():
    with open(os.path.join("test_images", name), "wb") as f:
        f.write(content)
print("Test images:", os.listdir("test_images"))

## 5. Run the pipeline

This does: load → inspect (shape, class order, preprocessing) → convert (quantized, with fallbacks) → validate (Keras vs TFLite top-1) → write `cabbageguard.tflite`, `labels.json`, `MODEL_NOTES.md`.

In [ ]:
import runpy

args = [sys.argv[0], MODEL_PATH]
if CLASSES:
    args += ["--classes", CLASSES]
if PREPROCESSING:
    args += ["--preprocessing", PREPROCESSING]
args += ["--tests", "test_images"]
sys.argv = args
runpy.run_path("cabbageguard_to_tflite.py", run_name="__main__")

## 6. Inspect the deliverables

In [ ]:
print(open("labels.json").read())
print("=" * 60)
print(open("MODEL_NOTES.md").read())

## 7. Download

Then in GreenBidder:
1. Copy `cabbageguard.tflite` and `labels.json` into `src/models/` (overwrite the placeholders)
2. Copy `MODEL_NOTES.md` to `docs/cabbage-feature/MODEL_NOTES.md`
3. Rebuild the native app — no code changes needed

In [ ]:
files.download("cabbageguard.tflite")
files.download("labels.json")
files.download("MODEL_NOTES.md")